# Unit 5: Robust Code — Errors & Files

> **Python Fundamentals**

Complete the reading materials and curated videos before working through this notebook.


## Learning objectives

By the end of this unit you will be able to:

- Use try/except to handle errors without crashing
- Read a CSV file into a list of dictionaries using csv.DictReader
- Write a complete data pipeline: read, process, flag, report


## 1. When things go wrong

So far, our code has assumed everything works perfectly. But real data is messy: files are missing, numbers are formatted as text, users type the wrong thing. When Python hits something it can't handle, it raises an **exception** and — unless you handle it — the whole program crashes.


In [ ]:
prices = ["10", "25", "oops", "40"]

total = 0
for price in prices:
    total += int(price)  # this will crash on "oops"

print(f"Total: {total}")


## 2. `try` / `except` — handling errors gracefully

Wrap risky code in a `try` block. If an exception occurs, Python jumps to the matching `except` block instead of crashing — and the rest of the program keeps running.


In [ ]:
prices = ["10", "25", "oops", "40"]

total = 0
skipped = 0
for price in prices:
    try:
        total += int(price)
    except ValueError:
        print(f"Skipping invalid price: {price!r}")
        skipped += 1

print(f"\nTotal: {total}, skipped {skipped} invalid value(s)")


### `else` and `finally`

- The `else` block runs only if the `try` block did **not** raise an exception.
- The `finally` block always runs, whether or not an exception occurred — useful for clean-up code (e.g. closing a file or connection).


In [ ]:
def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print(f"Cannot divide {a} by zero")
        return None
    else:
        print(f"{a} / {b} = {result}")
        return result
    finally:
        print("  (division attempt finished)\n")


safe_divide(10, 2)
safe_divide(10, 0)


## 3. Reading and writing files with `with`

Use `open()` together with a `with` block to read or write files. The `with` block automatically closes the file when you're done — even if an error occurs partway through.

Let's create a small CSV file to work with for the rest of this unit: a week of café sales transactions.


In [ ]:
csv_text = """date,item,quantity,unit_price
2026-02-02,Coffee,3,4.50
2026-02-02,Tea,1,3.50
2026-02-02,Sandwich,2,8.00
2026-02-03,Coffee,5,4.50
2026-02-03,Muffin,4,3.00
2026-02-04,Coffee,2,4.50
2026-02-04,Sandwich,1,8.00
2026-02-04,Tea,abc,3.50
2026-02-05,Coffee,6,4.50
2026-02-05,Muffin,3,3.00
"""

with open("sales.csv", "w") as f:
    f.write(csv_text)

print("sales.csv written.")


Notice the line `2026-02-04,Tea,abc,3.50` — the quantity is the text `abc` instead of a number. This is deliberate: real data is rarely perfectly clean, and our pipeline needs to handle it without crashing.

## 4. Reading CSV files with `csv.DictReader`

The `csv` module's `DictReader` reads each row of a CSV file straight into a **dict**, using the header row as the keys. This connects directly back to the list-of-dicts pattern from Unit 4.


In [ ]:
import csv

with open("sales.csv") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

print(f"Read {len(rows)} rows.\n")
print(rows[0])
print(rows[1])


Every value from `DictReader` is a **string** — even `quantity` and `unit_price`, which look like numbers. You'll need to convert them with `int()` or `float()` before doing arithmetic, using `try`/`except` to handle rows where that conversion fails.

## 5. Writing CSV files with `csv.DictWriter`

`DictWriter` is the mirror image of `DictReader` — it writes a list of dicts back out to a CSV file, given a list of column names (`fieldnames`).


In [ ]:
summary_rows = [
    {"item": "Coffee", "total_revenue": 72.00},
    {"item": "Tea", "total_revenue": 3.50},
    {"item": "Sandwich", "total_revenue": 24.00},
    {"item": "Muffin", "total_revenue": 21.00},
]

with open("sales_summary.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["item", "total_revenue"])
    writer.writeheader()
    writer.writerows(summary_rows)

with open("sales_summary.csv") as f:
    print(f.read())


## 6. The monthly P&L pipeline pattern

Most real-world data tasks follow the same shape:

1. **Read** the raw data (from a CSV)
2. **Process** each row — convert types, calculate derived values — skipping/flagging rows that fail (using `try`/`except`)
3. **Flag** anything unusual (e.g. very large transactions, invalid rows)
4. **Report** a summary (totals, counts, a written-out report file)

Let's build this pipeline for the café sales data.


In [ ]:
import csv

clean_rows = []
skipped_rows = []

# 1. Read
with open("sales.csv") as f:
    reader = csv.DictReader(f)

    # 2. Process — convert types, skipping bad rows
    for row in reader:
        try:
            quantity = int(row["quantity"])
            unit_price = float(row["unit_price"])
        except ValueError:
            skipped_rows.append(row)
            continue

        clean_rows.append({
            "date": row["date"],
            "item": row["item"],
            "quantity": quantity,
            "unit_price": unit_price,
            "line_total": quantity * unit_price,
        })

print(f"Processed {len(clean_rows)} valid rows, skipped {len(skipped_rows)}.")
for bad in skipped_rows:
    print(f"  Skipped: {bad}")


In [ ]:
# 3. Flag — transactions over a threshold
flag_threshold = 20.00
flagged = [row for row in clean_rows if row["line_total"] > flag_threshold]

print(f"Flagged {len(flagged)} large transaction(s):")
for row in flagged:
    date = row["date"]
    item = row["item"]
    line_total = row["line_total"]
    print(f"  {date} - {item}: ${line_total:.2f}")


In [ ]:
# 4. Report — totals per item, written to a summary file
totals_by_item = {}
for row in clean_rows:
    item = row["item"]
    totals_by_item[item] = totals_by_item.get(item, 0) + row["line_total"]

grand_total = sum(totals_by_item.values())

with open("sales_summary.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["item", "total_revenue"])
    writer.writeheader()
    for item, total in totals_by_item.items():
        writer.writerow({"item": item, "total_revenue": round(total, 2)})

print("Revenue by item:")
for item, total in totals_by_item.items():
    print(f"  {item}: ${total:.2f}")
print(f"\nGrand total: ${grand_total:.2f}")
print("Written to sales_summary.csv")


## 7. Putting it together

The CSV file `sales.csv` (created earlier in this notebook) also contains a row for **2026-02-03** where `Muffin` quantity is `4`. Extend the pipeline above to also calculate, **per date**:

1. The total revenue for that date
2. The number of valid (non-skipped) transactions on that date

Print one line per date, e.g. `2026-02-02: $25.50 across 3 transactions`.

*Hint: re-read `sales.csv` with `csv.DictReader`, reuse the same try/except pattern from section 6, and use a dict keyed by date to accumulate totals and counts.*


In [ ]:
import csv

# TODO: read sales.csv, convert types with try/except,
# and accumulate total revenue + transaction count per date



## Next steps

Open **`exercises.ipynb`** in this repository to practise what you've learned in this unit.
